# Rainfall nowcast prototype

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yanaput/DSA5208-rainfall/blob/main/prototype.ipynb)

Estimates **P(rainfall at a station in the next 30 minutes > τ mm)** with the trained model: one gradient-boosted tree
model that takes the threshold τ as an input, calibrated with isotonic regression on 2023 (see the report, §3).

Everything it needs is in this repository, so it runs on Colab without a cloud bucket:

- `prototype/models/`: the GBT, the isotonic calibrator and the climatology counts from the training years.
- `prototype/features/`: the model's inputs for every station and 5-minute time in **2024**, the held-out test year.

It uses the same prediction code as the pipeline (`scripts/predict.py`), run on PySpark in local mode. Nothing is
trained here. Run the cells from top to bottom; the first one takes about a minute on Colab.

In [ ]:
# Setup: on Colab, get the repository, Java 17 and PySpark. Locally, run from the repository root instead.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("prototype"):
        if not os.path.isdir("DSA5208-rainfall"):
            subprocess.run(["git", "clone", "-q", "https://github.com/Yanaput/DSA5208-rainfall.git"], check=True)
        os.chdir("DSA5208-rainfall")
    java = "/usr/lib/jvm/java-17-openjdk-amd64"   # Spark 4 needs Java 17
    if not os.path.isdir(java):
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless", shell=True, check=True)
    os.environ["JAVA_HOME"] = java
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pyspark==4.0.1"], check=True)
else:
    os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Homebrew's Java 17 on macOS

sys.path.insert(0, "scripts")
print("working directory:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

from predict import predict, score
from train import GRID_STEPS

ROOT = "prototype"   # holds models/ and features/
# Times are Singapore time. On a UTC session the hour and month features would shift by 8 hours.
spark = (SparkSession.builder.master("local[*]").appName("Rainfall nowcast prototype")
         .config("spark.sql.session.timeZone", "Asia/Singapore")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
features = spark.read.parquet(f"{ROOT}/features")


def rows_at(time, station=None):
    """Feature rows at a 5-minute time (Singapore time), for one station or all of them."""
    rows = features.where(F.col("bucket") == F.to_timestamp(F.lit(time)))
    return rows if station is None else rows.where(F.col("station_id") == station)


stations = sorted(r.station_id for r in features.select("station_id").distinct().collect())
print(f"{len(stations)} stations with 2024 data:", ", ".join(stations))

## 1. Ask the model

Choose a station, a time in 2024 (Singapore time, on a 5-minute mark) and a threshold. A prediction needs the station
to have reported at that time and to have complete readings for the following 30 minutes, which is how its accuracy
was measured; otherwise the cell says so. The threshold is rounded down to the 0.2 mm grid, which is exact because
readings come in 0.2 mm steps.

In [ ]:
STATION = "S116"  #@param {type:"string"}
TIME = "2024-01-24 15:30"  #@param {type:"string"}
THRESHOLD_MM = 5  #@param {type:"number"}

r = predict(spark, ROOT, STATION, TIME, THRESHOLD_MM)
row = rows_at(TIME, STATION).first()
print(f"P(more than {r.threshold:g} mm at {STATION} in the 30 minutes after {r.bucket:%Y-%m-%d %H:%M}) = {r.cal:.1%}")
print(f"In the previous 30 minutes: {row.rain_30m:g} mm.  What actually fell in the next 30 minutes: "
      f"{row.rain_next_30m:g} mm.")

## 2. Every threshold at once

Because the threshold is an input, one row gives the whole curve P(rain > τ) for τ = 0, 0.2, …, 30 mm. The dashed line
is the rain that actually fell.

In [ ]:
grid = [j / 5 for j in range(GRID_STEPS + 1)]   # 0, 0.2, ..., 30 mm
curve = score(spark, ROOT, rows_at(TIME, STATION), grid).select("threshold", "cal").toPandas()

fig, ax = plt.subplots(figsize=(7, 4))
ax.step(curve.threshold, curve.cal, where="post", label="P(rain in next 30 min > τ)")
ax.axvline(row.rain_next_30m, color="tab:red", ls="--", label=f"observed: {row.rain_next_30m:g} mm")
# The grid ends at 30 mm: larger thresholds get the 30 mm answer.
ax.set(xlabel="threshold τ (mm)", ylabel="probability", ylim=(0, 1.02), xlim=(0, max(30, row.rain_next_30m + 2)),
       title=f"{STATION}, {TIME}")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 3. The whole island at that time

The same threshold at every station reporting at that time. Rings mark stations where the threshold was actually
exceeded.

In [ ]:
island = (score(spark, ROOT, rows_at(TIME), [THRESHOLD_MM])
          .select("station_id", "lat", "lon", "cal", "rain_next_30m").toPandas())
hit = island.rain_next_30m > r.threshold

fig, ax = plt.subplots(figsize=(8, 5))
points = ax.scatter(island.lon, island.lat, c=island.cal, cmap="Blues", vmin=0, vmax=1, s=80, edgecolor="grey")
ax.scatter(island.lon[hit], island.lat[hit], s=220, facecolor="none", edgecolor="tab:red", lw=1.5,
           label=f"observed > {r.threshold:g} mm")
for s in island.itertuples():
    ax.annotate(s.station_id, (s.lon, s.lat), xytext=(4, 4), textcoords="offset points", fontsize=6)
fig.colorbar(points, ax=ax, label=f"P(> {r.threshold:g} mm in next 30 min)")
ax.set(xlabel="longitude", ylabel="latitude", title=f"{TIME}, {len(island)} stations reporting")
ax.set_aspect("equal")
ax.legend(loc="lower right")
plt.show()

## 4. Through the day at the station

The forecast at every 5-minute issue time of that day, at the chosen threshold and at 1 mm, over the rain that
actually fell in the following 30 minutes.

In [ ]:
day = features.where((F.col("station_id") == STATION) & (F.to_date("bucket") == TIME[:10]))
ts = (score(spark, ROOT, day, sorted({1.0, float(r.threshold)}))
      .select("bucket", "threshold", "cal", "rain_next_30m").toPandas().sort_values("bucket"))

fig, ax = plt.subplots(figsize=(11, 4))
rain = ax.twinx()
obs = ts.drop_duplicates("bucket")
rain.bar(obs.bucket, obs.rain_next_30m, width=1 / 288, color="lightgrey", label="observed rain, next 30 min")
rain.set_ylabel("observed rain in next 30 min (mm)")
for t, g in ts.groupby("threshold"):
    ax.plot(g.bucket, g.cal, label=f"P(> {t:g} mm)")
ax.set_zorder(rain.get_zorder() + 1)   # probabilities in front of the bars
ax.patch.set_visible(False)
ax.set(ylabel="probability", ylim=(0, 1.02), title=f"{STATION}, {TIME[:10]}")
ax.legend(loc="upper left")
rain.legend(loc="upper right")
plt.show()